# Excel 포맷팅 및 시트 전환 테스트

여러 시트에 다양한 Excel 포맷팅을 적용하고 시트 간 전환하며 테스트합니다.

In [11]:
import pandas as pd
import numpy as np
from dh_tool.dataframe import Sheets
import tempfile
import os
import pandas as pd
import numpy as np
from dh_tool.dataframe import Sheets
import tempfile
import os
from openpyxl.styles import PatternFill, Font  # 이 부분 추가

In [12]:
# 특이한 데이터를 포함한 데이터프레임 생성
df1 = pd.DataFrame({
    'URL': ['http://example.com', 'http://test.com'],
    'LongText': ['A' * 1000, 'B' * 1000],
    'SpecialChars': ['한글★☆', '特殊文字'],
    'Numbers': [1234.5678, -9876.5432]
})

df2 = pd.DataFrame({
    'Nested': [{'a': 1, 'b': 2}, {'c': 3, 'd': 4}],
    'Arrays': [np.array([1, 2, 3]), np.array([4, 5, 6])],
    'Mixed': [[1, 'a'], [2, 'b']],
    'Dates': [pd.Timestamp('2023-01-01'), pd.Timestamp('2023-12-31')]
})

# Sheets 객체 생성 및 시트 추가
sheets = Sheets(df1)
sheets.create_sheet(df2, "특수문자시트★")

In [16]:
with pd.ExcelWriter("test_z.xlsx") as writer:    
    df1.to_excel(writer, sheet_name='test')

In [15]:
df2.to_excel('test_z.xlsx', index=False)

In [3]:
sheets

,Nested,Arrays,Mixed,Dates
0,"{'a': 1, 'b': 2}",[1 2 3],"[1, 'a']",2023-01-01
1,"{'c': 3, 'd': 4}",[4 5 6],"[2, 'b']",2023-12-31


In [4]:
sheets.sheet_names

['Sheet1', '특수문자시트★']

In [5]:
# Sheet1에 다양한 포맷팅 적용
sheets.select_sheet("Sheet1")
sheets.excel_handler.freeze_first_row()
sheets.excel_handler.enable_autowrap()
sheets.excel_handler.set_column_width(URL=30, LongText=50, SpecialChars=15, Numbers=15)

# URL 열에 하이퍼링크 추가
urls = df1['URL'].tolist()
sheets.excel_handler.add_hyperlinks_to_column('URL', urls)

# Numbers 열에 조건부 서식 추가 (음수는 빨간색)
sheets.excel_handler.add_conditional_formatting(
    "D2:D100",  # Numbers 열 범위
    "D2<0",
    PatternFill(start_color="FFFF0000", end_color="FFFF0000", fill_type="solid")
)

print("Sheet1 포맷팅 완료")

Sheet1 포맷팅 완료


In [6]:
# 특수문자시트에 포맷팅 적용

sheets.select_sheet("특수문자시트★")
sheets.excel_handler.freeze_first_row()
sheets.excel_handler.auto_adjust_columns()

# 사용자 정의 스타일 함수 적용
def custom_style(cell):
    if isinstance(cell.value, dict):
        cell.font = Font(color="FF0000")
    elif isinstance(cell.value, str) and cell.value.startswith('['):
        cell.font = Font(color="0000FF")

sheets.excel_handler.apply_style(custom_style)

print("특수문자시트★ 포맷팅 완료")

특수문자시트★ 포맷팅 완료


In [7]:
# 시트 간 전환하며 데이터 확인
for sheet_name in sheets.sheet_names:
    print(f"\n{sheet_name} 데이터:")
    sheets.select_sheet(sheet_name)
    display(sheets.df)


Sheet1 데이터:


,URL,LongText,SpecialChars,Numbers
0,http://example.com,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,한글★☆,1234.5678
1,http://test.com,BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBB...,特殊文字,-9876.5432



특수문자시트★ 데이터:


,Nested,Arrays,Mixed,Dates
0,"{'a': 1, 'b': 2}",[1 2 3],"[1, 'a']",2023-01-01
1,"{'c': 3, 'd': 4}",[4 5 6],"[2, 'b']",2023-12-31


In [8]:
# 임시 파일에 저장 및 확인
with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
    try:
        sheets.save(tmp.name)
        print(f"파일 저장 성공: {tmp.name}")
        
        # 저장된 파일 다시 로드해서 확인
        loaded_sheets = Sheets(pd.read_excel(tmp.name))
        
        for sheet_name in loaded_sheets.sheet_names:
            print(f"\n로드된 {sheet_name} 데이터:")
            loaded_sheets.select_sheet(sheet_name)
            display(loaded_sheets.df)
            
    except Exception as e:
        print(f"에러 발생: {str(e)}")
    finally:
        os.unlink(tmp.name)

파일 저장 성공: /tmp/tmp458corw3.xlsx

로드된 Sheet1 데이터:


,URL,LongText,SpecialChars,Numbers
0,http://example.com,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,한글★☆,1234.5678
1,http://test.com,BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBB...,特殊文字,-9876.5432


In [9]:
sheets.save('tmp.xlsx')

In [10]:
# 시트 삭제 테스트
sheets.remove_sheet("특수문자시트★")
print(f"남은 시트 목록: {sheets.sheet_names}")

# 마지막 시트 선택
sheets.select_sheet("Sheet1")
print(f"현재 활성 시트: {sheets.current_sheet}")
display(sheets.df)

남은 시트 목록: ['Sheet1']
현재 활성 시트: Sheet1


,URL,LongText,SpecialChars,Numbers
0,http://example.com,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,한글★☆,1234.5678
1,http://test.com,BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBB...,特殊文字,-9876.5432
